## LIBRARIES AND DECLARATION

In [ ]:
import os
import time
import pandas as pd
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
import json
from IPython.display import FileLink

## CONFIG PROVIDER

In [ ]:
secrets = UserSecretsClient()
NINE_ROUTER_API_KEY = secrets.get_secret("NINE_ROUTER_API_KEY")
NINE_ROUTER_BASE_URL = "https://r8slvik.abc-tunnel.us/v1"

MODEL_CONFIG = {
    "model": "cx/gpt-5.5",
    "api_key": NINE_ROUTER_API_KEY,
    "base_url": NINE_ROUTER_BASE_URL,
}

INPUT_FILE = "/kaggle/input/datasets/chauanhh/data-test/data-test.csv"
CHECKPOINT_FILE = "/kaggle/working/data-test_judge.csv"

BATCH_SIZE = 6
BATCH_PAUSE = 30
ROW_PAUSE = 4
MAX_RETRIES = 3

client = OpenAI(
    api_key=MODEL_CONFIG["api_key"],
    base_url=MODEL_CONFIG["base_url"],
)

## HELPERS

In [ ]:
def clean_code_response(text):
    if text is None:
        return ""

    text = text.strip()

    if text.startswith("```cpp"):
        text = text.replace("```cpp", "", 1).strip()
    elif text.startswith("```c++"):
        text = text.replace("```c++", "", 1).strip()
    elif text.startswith("```"):
        text = text.replace("```", "", 1).strip()

    if text.endswith("```"):
        text = text[:-3].strip()

    return text.strip()

In [ ]:
def call_llm(prompt, **kwargs):
    system = kwargs.pop("system", None)

    params = {
        "model": MODEL_CONFIG["model"],
        "max_tokens": kwargs.pop("max_tokens", 4096),
        "messages": [{"role": "user", "content": prompt}],
    }
    if system:
        params["system"] = system
    params.update(kwargs)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.messages.create(**params)
            content = response.content[0].text
            if not content:
                raise ValueError("Model returned empty response")
            return content.strip()
        except Exception as e:
            print(f"  API error (attempt {attempt}/{MAX_RETRIES}): {e}")
            if attempt < MAX_RETRIES:
                print("  Retrying in 20s...")
                time.sleep(20)
    return None

In [ ]:
BUG_LINES = [
    "- Uninitialized variable - e.g. `int x;` used in computation without assignment ",
    "- Off-by-one index - e.g. accessing `a[n]` instead of `a[n-1]`, or using `<=` instead of `<` in loop bounds ",
    "- Wrong condition operator - e.g. replacing `!=` with `==`",
    "- Wrong arithmetic operator - e.g. using `m >> 1` instead of `m << 1`",
    "- Wrong variable name - e.g. using `i0` instead of `i1`",
    "- Missing edge case handling - e.g. missing handling for negatives, duplicates, or disconnected graph",
    "Assignment instead of comparison - e.g. `if (i = 0)` instead of `if (i == 0)",
    "Fake optimization - e.g. adding unnecessary `vector.clear()", 
    "Logical/bitwise operator confusion - e.g. using `&` instead of `&&`, or `|` instead of `||",
    "Misleading comment - e.g. comment describes behavior that differs from what the code actually does",
    "Insufficient recursion base case - e.g. missing termination condition causing stack overflow",
    "Operator precedence error - e.g. `mid = l + r / 2` instead of `mid = l + (r - l) / 2",
    "Forgetting to reset global/static state  - e.g. global visited array not cleared between test cases",
]


def _generate_single_bug(original_solution, bug_line, max_retries=2):
    noise_prompt = f"""
### You will be given a correct C++ solution to a coding problem.
Submission:
```cpp
{original_solution}
```
### Your task is to modify the submission by introducing exactly one as described below.
{bug_line}
---
### Step 2: Apply the bug and describe the modification (1-sentence description)
Use the format:
  [<bug name>] <short description of what was changed> 
Note: Do NOT change any other logic. You may lightly reformat the surrounding code (brace style, line breaks) but the algorithm must remain identical. 
---
### Step 3: Output the buggy submission
Apply the selected bug to the submission with minimum modification to the original code.
---
### Respond with a JSON object only:
{{
  "bug_description": "[<name>] <description>",
  "buggy_submission": "<source code with the modification>"
}}
"""
    for attempt in range(max_retries + 1):
        try:
            response = client.responses.create(
                model=MODEL_CONFIG["model"],
                input=noise_prompt,
            )
            output = response.output_text
        except Exception as error:
            print(f"API error: {error}")
            continue

        if not output:
            continue

        output_clean = output.strip()

        if output_clean.startswith("```"):
            lines = output_clean.splitlines()
            if lines and lines[0].strip().lower() in ("```", "```json"):
                lines = lines[1:]
            if lines and lines[-1].strip() == "```":
                lines = lines[:-1]
            output_clean = "\n".join(lines).strip()

        try:
            parsed = json.loads(output_clean)
        except json.JSONDecodeError as error:
            print(f"JSON parsing error: {error}")
            print("Raw output:")
            print(output)
            continue

        bug_description = parsed.get("bug_description", "")
        raw_code = parsed.get("buggy_submission", "")
        buggy_code = clean_code_response(raw_code)

        if not buggy_code:
            continue

        return {
            "bug_description": bug_description,
            "buggy_submission": buggy_code,
        }

    return None


def generate_buggy_code(problem, original_solution):
    records = []
    for bug_line in BUG_LINES:
        result = _generate_single_bug(original_solution, bug_line)
        if result is None:
            print(f" Skipping #bug: {bug_line.strip()}")
            continue
        records.append(result)
    return records

## LOAD CHECKPOINT

In [ ]:
if os.path.exists(CHECKPOINT_FILE):
    done_df = pd.read_csv(CHECKPOINT_FILE)
    done_ids = set(done_df["id"].astype(str).tolist())
    results = done_df.to_dict("records")

    print(f"Resumed: {len(done_ids)} records already processed.")
    
else:
    done_ids = set()
    results = []
    
print("Starting fresh...")

In [ ]:
print("Reading input file...")
data = pd.read_csv(INPUT_FILE)

# random ids
data.rename(columns={"buggy_submission": "solution"}, inplace=True)
ids1 = [345771672, 344951457, 345028393, 345041453, 345049199]
id2 = [344951457, 345028393, 345041453, 345049199]
ids = [345771672, 345049199]

df = data[data["id"].isin(ids)]

pending_df = df[~df["id"].astype(str).isin(done_ids)].reset_index(drop=True)
total = len(pending_df)
batches = [pending_df.iloc[i:i + BATCH_SIZE] for i in range(0, total, BATCH_SIZE)]

print(f"Pending: {total} records → {len(batches)} batches (size={BATCH_SIZE})")

In [ ]:
for batch_idx, batch in enumerate(batches):
    print(f"\n--- Batch {batch_idx + 1}/{len(batches)} ---")

    for _, row in batch.iterrows():
        row_id = row["id"]
        topic = row.get("topic", "")
    
        print(f"Processing ID {row_id} - Topic: {topic}...")
    
        buggy_versions = generate_buggy_code(
            problem=row["problem"],
            original_solution=row["solution"]
        )
    
        if buggy_versions:
            for bug in buggy_versions:
                row_dict = row.to_dict()
                row_dict["buggy_submission"] = bug["buggy_submission"]
                row_dict["bug_description"] = bug["bug_description"]
                results.append(row_dict)
    
            pd.DataFrame(results).to_csv(
                CHECKPOINT_FILE,
                index=False,
                encoding="utf-8-sig"
            )
    
            print(f"  Checkpoint saved ({len(results)} records)")
        else:
            print(f"  Failed to generate buggy code for ID {row_id}, skipping...")
    
        time.sleep(ROW_PAUSE)

    # Pause between batches
    if batch_idx < len(batches) - 1:
        print(f"\nBatch {batch_idx + 1} done. Pausing {BATCH_PAUSE}s...")
        time.sleep(BATCH_PAUSE)

print(f"\nDone! Saved {len(results)} records to {CHECKPOINT_FILE}")

In [ ]:
# --- CHECK MISSING IDs ---
print("Checking missing IDs...")
data = pd.read_csv(INPUT_FILE)

#random ids
data.rename(columns={"buggy_submission": "solution"}, inplace=True)
ids = [345771672, 344951457, 345028393, 345041453, 345049199]
df = data[data["id"].isin(ids)]

done_df = pd.read_csv(CHECKPOINT_FILE)

all_ids = set(df['id'].astype(str).tolist())
done_ids = set(done_df['id'].astype(str).tolist())
missing_ids = all_ids - done_ids

if not missing_ids:
    print("No missing IDs, dataset is complete!")
else:
    print(f"Found {len(missing_ids)} missing IDs: {sorted(missing_ids)}")
    missing_df = df[df['id'].astype(str).isin(missing_ids)].reset_index(drop=True)
    results = done_df.to_dict('records')

    for _, row in missing_df.iterrows():
        print(f"Retrying missing ID {row['id']} - Topic: {row['topic']}...")

        buggy_versions = generate_buggy_code(row['problem'], row['solution'])

        if buggy_versions:
            for bug in buggy_versions:
                row_dict = row.to_dict()
                row_dict['buggy_submission'] = bug['buggy_submission']
                row_dict['bug_description'] = bug['bug_description']
                results.append(row_dict)

            # Sort by id to maintain order before saving
            results_df = pd.DataFrame(results).sort_values('id').reset_index(drop=True)
            results_df.to_csv(CHECKPOINT_FILE, index=False, encoding='utf-8-sig')
            print(f"  Checkpoint saved ({len(results)} records)")
        else:
            print(f"  Failed to generate buggy code for ID {row['id']}, skipping...")

        time.sleep(4)

    print(f"\nDone! Saved {len(results)} records to {CHECKPOINT_FILE}")

In [ ]:
FileLink(r'CHECKPOINT_FILE')